# Can we predict late delivery?
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), walk-forward CV (5 splits) on training rows only, choice by mean CV roc_auc, holdout scored once.

**Feature decision:** `claim_paid_eur` is compensation paid on a claim settled for the consignment, i.e. recorded after the delivery outcome (it is > 0 almost only when late). It is not known at dispatch, so it is excluded as leakage. `consignment` (id) and `dispatched_on` (split column) are excluded per protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
df = pd.read_csv('data.csv')
hold_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(hold_idx)==1610
train_idx = np.setdiff1d(np.arange(len(df)), hold_idx)
# walk-forward needs training rows in time order
train_idx = train_idx[np.argsort(pd.to_datetime(df.loc[train_idx,'dispatched_on']).to_numpy(), kind='stable')]
print(len(train_idx), len(hold_idx))

In [ ]:
target='late_delivery'
cat=['carrier']; num=['distance_km','weight_kg','parcels']
features=cat+num   # claim_paid_eur excluded: recorded after the outcome
X=df[features]; y=df[target]
Xtr,ytr=X.loc[train_idx],y.loc[train_idx]

In [ ]:
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),
                              ('n',StandardScaler() if scale else 'passthrough',num)])
# all candidates named before any scoring
candidates={
 'baseline_prior': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=1,max_iter=1000))]),
 'rf_depth6': Pipeline([('p',prep(False)),('m',RandomForestClassifier(300,max_depth=6,min_samples_leaf=20,random_state=0))]),
 'hgb_small': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,random_state=0))]),
}
print(list(candidates))

In [ ]:
tscv=TimeSeriesSplit(n_splits=5)
folds=list(tscv.split(Xtr))
# row labels = 0-based positions in data.csv
cv_folds=[[[int(i) for i in train_idx[a]],[int(i) for i in train_idx[b]]] for a,b in folds]
cv_scores={}
for name,m in candidates.items():
    s=[]
    for a,b in folds:
        mm=clone(m).fit(Xtr.iloc[a],ytr.iloc[a])
        s.append(float(roc_auc_score(ytr.iloc[b],mm.predict_proba(Xtr.iloc[b])[:,1])))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); print(cv.round(4))
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k])); print('chosen:',chosen)

In [ ]:
final=clone(candidates[chosen]).fit(Xtr,ytr)
Xh,yh=X.loc[hold_idx],y.loc[hold_idx]
holdout_auc=float(roc_auc_score(yh,final.predict_proba(Xh)[:,1]))
print('holdout roc_auc (scored once):',round(holdout_auc,4))

In [ ]:
evidence={'features':features,'candidates':list(candidates),'cv_folds':cv_folds,'cv_scores':cv_scores,
 'chosen':chosen,'holdout_rows':[int(i) for i in hold_idx],'holdout_scores':{chosen:{'roc_auc':holdout_auc}}}
json.dump(evidence,open('evidence.json','w'))